In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/content/drive/MyDrive/capstone/dataset/Dataset ocr 1.multiclass/dataset_transaksi.csv')

df.head()

,transaction_id,tanggal,nama_produk,qty,harga_satuan,total_item,total_transaksi
0,X51005705729_jpg,13/10/2017,Timne:,1,11.37,0.00,NaN
1,X51005705727_jpg,28/12/2017,Time:,1,10.50,0.00,NaN
2,X51005442386_jpg,NaN,GST @,8,6.00,50.49,58.7
3,X51005757201_jpg,28/02/2018,DISC,1,10.00,8.01,NaN
4,X51005757201_jpg,28/02/2018,OISC',1,10.00,2.25,NaN


In [4]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

df.columns

Index(['transaction_id', 'tanggal', 'nama_produk', 'qty', 'harga_satuan',
       'total_item', 'total_transaksi'],
      dtype='object')

In [8]:
# cek missing value
print(df.isnull().sum())

# hapus data penting yang kosong
df = df.dropna(subset=["transaction_id", "nama_produk"])

# isi nilai kosong (sesuai nama kolom asli)
df["qty"] = df["qty"].fillna(1)
df["harga_satuan"] = df["harga_satuan"].fillna(0)

# opsional: total_transaksi
df["total_transaksi"] = df["total_transaksi"].fillna(df["total_item"])

transaction_id      0
tanggal            17
nama_produk         0
qty                 0
harga_satuan        0
total_item          0
total_transaksi     0
dtype: int64


In [14]:
# ubah tipe data yang benar
df["qty"] = df["qty"].astype(int)
df["harga_satuan"] = df["harga_satuan"].astype(float)

# tanggal
df["tanggal"] = pd.to_datetime(df["tanggal"], errors='coerce')

In [15]:
df["nama_produk"] = df["nama_produk"].str.lower().str.strip()

# hapus karakter aneh
df["nama_produk"] = df["nama_produk"].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)

In [16]:
df = df.drop_duplicates()

In [18]:
# pastikan tipe data aman
df["qty"] = pd.to_numeric(df["qty"], errors='coerce').fillna(1)
df["harga_satuan"] = pd.to_numeric(df["harga_satuan"], errors='coerce').fillna(0)

# total
df["total"] = df["qty"] * df["harga_satuan"]

# tanggal
df["tanggal"] = pd.to_datetime(df["tanggal"], errors='coerce')

# fitur waktu
df["hari"] = df["tanggal"].dt.day_name()
df["bulan"] = df["tanggal"].dt.month

In [19]:
df.head()

,transaction_id,tanggal,nama_produk,qty,harga_satuan,total_item,total_transaksi,total,hari,bulan
0,X51005705729_jpg,2017-10-13,timne,1,11.37,0.00,0.00,11.37,Friday,10.0
1,X51005705727_jpg,2017-12-28,time,1,10.50,0.00,0.00,10.50,Thursday,12.0
2,X51005442386_jpg,NaT,gst,8,6.00,50.49,58.70,48.00,NaN,NaN
3,X51005757201_jpg,2018-02-28,disc,1,10.00,8.01,8.01,10.00,Wednesday,2.0
4,X51005757201_jpg,2018-02-28,oisc,1,10.00,2.25,2.25,10.00,Wednesday,2.0


In [20]:
df_transaksi = df.groupby("transaction_id").agg({
    "tanggal": "first",
    "total": "sum",
    "nama_produk": "count"
}).reset_index()

df_transaksi.rename(columns={
    "nama_produk": "jumlah_item"
}, inplace=True)

df_transaksi.head()

,transaction_id,tanggal,total,jumlah_item
0,X51005200931_jpg,2018-02-09,8.32,1
1,X51005230648_jpg,2018-01-29,6.00,1
2,X51005230657_jpg,2017-12-31,1.80,1
3,X51005433492_jpg,NaT,69.00,1
4,X51005433522_jpg,NaT,8.00,1


In [23]:
# buat market basket
basket = df.groupby(['transaction_id', 'nama_produk'])['qty'].sum().unstack().fillna(0)

# ubah ke 0/1
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

/tmp/ipykernel_11198/3802449073.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


In [25]:
df.to_csv('/content/dataset_clean.csv', index=False)
df_transaksi.to_csv('/content/dataset_transaksi.csv', index=False)
basket.to_csv('/content/dataset_market_basket.csv')

In [26]:
from google.colab import files

files.download('/content/dataset_clean.csv')
files.download('/content/dataset_transaksi.csv')
files.download('/content/dataset_market_basket.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>